In [ ]:
import dataset
from langchain_core.prompts import PromptTemplate
from langchain_ollama.llms import OllamaLLM
from langchain_classic.evaluation import ExactMatchStringEvaluator

def gen_model(model='gemma2:2b', template=None):
    if template is None:
        template = """
        Respond to the question with a single letter: A,B,C or D. Only output the letter of the correct answer.
        Question: {question}
        Answer:{answer}
        """.strip()

    prompt = PromptTemplate.from_template(template)
    # deterministic model
    model = OllamaLLM(model=model,temperature=0,num_predict=1,top_k=10,top_p=0.5)
    chain = prompt | model
    return lambda question: chain.invoke({'question': question,'answer': ''}).strip()

def gen_question(item):
    options = [f'{k}: {v}' for k,v in item['choices'].items()]
    question = item['question']
    return '\n'.join([question] + options)

def gen_evaluator():
    evaluator = ExactMatchStringEvaluator()
    return lambda ans, ref: evaluator.evaluate_strings(prediction=ans,reference=ref)['score']

def dataloader(path='data/questions.jsonl'):
    for item in (dataset.load_questions(path)):
        yield gen_question(item), item['answer']

def eval_model(model, dataset, print_questions=False, print_iterative_scores=False):
    scorer = gen_evaluator()
    ok = 0
    for i, (q, ref) in enumerate(dataset):
        ans = model(q)
        sc = score(ans, ref)
        if sc == 1.0: 
            ok += 1
        acc = round(100 * ok / (i+1), 2)

        if print_questions:
            print(q)
            print('answered:', ans, 'correct:', ref)
            print('--')
        if print_iterative_scores:
            print('round', i, '| acc:', acc)
            print('--')
    
    return acc 

In [ ]:
model = gen_model()

ds = dataloader('data/questions2.jsonl')
eval_model(model, ds, print_iterative_scores=True)


round 0 | acc: 0.0
--
round 1 | acc: 0.0
--
round 2 | acc: 0.0
--
round 3 | acc: 25.0
--
round 4 | acc: 40.0
--
round 5 | acc: 50.0
--
round 6 | acc: 57.14
--
round 7 | acc: 50.0
--
round 8 | acc: 44.44
--
round 9 | acc: 40.0
--
round 10 | acc: 36.36
--
round 11 | acc: 33.33
--
round 12 | acc: 30.77
--
round 13 | acc: 28.57
--
round 14 | acc: 26.67
--
round 15 | acc: 31.25
--
round 16 | acc: 29.41
--
round 17 | acc: 27.78
--
round 18 | acc: 31.58
--
round 19 | acc: 35.0
--
round 20 | acc: 33.33
--
round 21 | acc: 31.82
--
round 22 | acc: 34.78
--
round 23 | acc: 33.33
--
round 24 | acc: 32.0
--


32.0